# Combat state print (notebook)

Same checks as `tests/test_combat_state_print.py`: load `combat_first_round.jkr`, advance to combat, run `print_combat_state`, and verify `_format_card_id` for a few IDs.

Run cells top to bottom. On Google Colab, the first code cell mounts Drive and `chdir`s to the repo (set `BALATRO_RL_AGENT_DIR` if your clone path differs).

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore[import-not-found]
except ImportError:
    drive = None

_DEFAULT_COLAB_REPO = (
    "/content/drive/MyDrive/Duke/CS 590 RL/Project/balatro-rl-agent"
)
if drive is not None:
    drive.mount("/content/drive")
    _colab_repo = os.environ.get("BALATRO_RL_AGENT_DIR", _DEFAULT_COLAB_REPO).rstrip("/")
    if os.path.isdir(_colab_repo):
        os.chdir(_colab_repo)
        if _colab_repo not in sys.path:
            sys.path.insert(0, _colab_repo)
    else:
        print(
            f"[CombatStatePrint] Skip Colab chdir: not a directory: {_colab_repo!r}. "
            "Set BALATRO_RL_AGENT_DIR to your clone path.",
            file=sys.stderr,
        )


def _repo_root() -> Path:
    """Notebook has no ``__file__``; infer repo root from cwd (after Colab ``chdir``)."""
    cwd = Path.cwd().resolve()
    if (cwd / "cs590_env").is_dir():
        return cwd
    if (cwd.parent / "cs590_env").is_dir():
        return cwd.parent
    return cwd


REPO_ROOT = _repo_root()
print("REPO_ROOT =", REPO_ROOT)

In [ ]:
import io

from balatro_gym.save_injection import inject_save_into_balatro_env

from cs590_env.combat_wrapper import CombatActionWrapper
from cs590_env.schema import GamePhase
from cs590_env.util import print_combat_state
from cs590_env.wrapper import BalatroPhaseWrapper

In [ ]:
COMBAT_JKR = REPO_ROOT / "save_files" / "combat" / "combat_first_round.jkr"


def _combat_obs_from_jkr(jkr_path: Path, seed: int = 42):
    """Build a COMBAT observation from an injected save without ``reset()`` (which would clear injection)."""
    base_env, _ = inject_save_into_balatro_env(jkr_path, seed=seed)
    phase = BalatroPhaseWrapper(base_env)
    phase._auto_skip_pack_open()
    obs = phase._get_phase_observation()
    combat = CombatActionWrapper(phase)
    obs = combat._advance_to_combat(obs)
    return obs


assert COMBAT_JKR.is_file(), f"Missing fixture: {COMBAT_JKR}"
print("Fixture:", COMBAT_JKR)

In [ ]:
# Equivalent to ``test_print_combat_state_from_jkr``
obs = _combat_obs_from_jkr(COMBAT_JKR, seed=42)
buf = io.StringIO()
print_combat_state(obs, file=buf)
text = buf.getvalue()

print("\n--- print_combat_state (fixture) ---", flush=True)
print(text, end="", flush=True)
print("--- end print_combat_state ---\n", flush=True)

assert "=== Run ===" in text
assert "=== Blind / score ===" in text
assert "=== Hand ===" in text
assert "=== Hand levels" in text
assert int(obs["phase"]) == int(GamePhase.COMBAT)
if int(obs["hand_size"]) > 0:
    assert any(c in text for c in "♣♦♥♠"), "expected rank+suit symbols in hand dump"
print("print_combat_state checks: OK")

In [ ]:
# Equivalent to ``test_format_card_id_known``
from cs590_env.util import _format_card_id

assert _format_card_id(0) == "2♣"
assert _format_card_id(51) == "A♠"
assert _format_card_id(-1) == "PAD"
print("_format_card_id checks: OK")